# L9.4 — Privacy, PII & Data Retention

Hands-on notebook for the lesson [`9-4-privacy.mdx`](../../llm-quest-theory/level-9/9-4-privacy.mdx).

> **Learning objectives**
> - Build a **PII redactor** that handles email, phone, national-ID, credit-card and IP patterns — with reversible tokens for debugging.
> - Track **consent** per user and refuse logging when consent is not granted.
> - Simulate a **retention policy** (delete logs older than `N` days).
> - Implement the **right to be forgotten** — delete every row belonging to one user across logs, memory, and cache.
> - Measure false-negative and false-positive rates of the regex redactor vs a hand-crafted test set.

## Connection to the theory
Covers **§1–§10** of the source `.mdx`. A production system would add ML-based PII detectors (Presidio, AWS Comprehend) on top of the regex layer.

In [1]:
# ---- Setup ----
import os, re, json, time, uuid, hashlib
from dataclasses import dataclass, asdict, field
from datetime import datetime, timedelta
import pandas as pd

SEED = 42

## 1. Regex-based PII redactor
Five common PII types. Each pattern returns a **stable token** keyed by a salted hash — so the same `alice@example.com` always maps to the same `[EMAIL_abc123]`, enabling debug while hiding the raw value.

In [2]:
SALT = b"llm-quest-privacy-salt"   # rotate per environment in production

def stable_token(kind, value):
    h = hashlib.sha1(SALT + value.lower().encode()).hexdigest()[:6]
    return f"[{kind}_{h}]"

PATTERNS = [
    ("EMAIL",  re.compile(r"\b[\w.+-]+@[\w-]+\.[\w.-]+\b")),
    ("PHONE",  re.compile(r"\b(?:\+?\d{1,3}[-.\s]?)?\(?\d{2,4}\)?[-.\s]?\d{3,4}[-.\s]?\d{3,4}\b")),
    ("SSN",    re.compile(r"\b\d{3}-\d{2}-\d{4}\b")),
    ("CREDIT", re.compile(r"\b(?:\d[ -]?){13,19}\b")),
    ("IP",     re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")),
]

def redact(text):
    substitutions = []
    for kind, pat in PATTERNS:
        def _sub(match):
            tok = stable_token(kind, match.group(0))
            substitutions.append((kind, match.group(0), tok))
            return tok
        text = pat.sub(_sub, text)
    return text, substitutions

# Demonstrate
sample = (
    "Hi Alice, please contact me at alice@example.com or 555-123-4567. "
    "My SSN is 123-45-6789 and my card 4111 1111 1111 1111. "
    "Our dev server is at 10.0.0.42."
)
redacted, rows = redact(sample)
print("RAW     :", sample)
print("REDACTED:", redacted)
print("subs    :", rows)

RAW     : Hi Alice, please contact me at alice@example.com or 555-123-4567. My SSN is 123-45-6789 and my card 4111 1111 1111 1111. Our dev server is at 10.0.0.42.
REDACTED: Hi Alice, please contact me at [EMAIL_88c15a] or [PHONE_2faa4f]. My SSN is [SSN_a65362] and my card [PHONE_5e7357] 1111. Our dev server is at [IP_05cfce].
subs    : [('EMAIL', 'alice@example.com', '[EMAIL_88c15a]'), ('PHONE', '555-123-4567', '[PHONE_2faa4f]'), ('PHONE', '4111 1111 1111', '[PHONE_5e7357]'), ('SSN', '123-45-6789', '[SSN_a65362]'), ('IP', '10.0.0.42', '[IP_05cfce]')]


## 2. Stability of the tokens
The same PII always maps to the same token — so if two logs mention the same email, a debugger can correlate them **without** seeing the raw address.

In [3]:
txt1 = "alice@example.com says hi"
txt2 = "also alice@example.com said hello"
r1, _ = redact(txt1)
r2, _ = redact(txt2)
print(r1)
print(r2)
# Both lines share the same [EMAIL_xxxxxx] token

[EMAIL_88c15a] says hi
also [EMAIL_88c15a] said hello


## 3. Accuracy test set — measure FP and FN

In [4]:
# Strings that **should** be redacted (positive)
POSITIVE = [
    "Ping jane.doe@example.co.uk",
    "Phone: +84 912 345 678",
    "SSN 078-05-1120 (sample)",
    "Credit: 5555 5555 5555 4444",
    "We scanned 192.168.0.1",
    "Combined: email=bob@bob.io phone=(123) 456 7890",
]

# Strings that should **not** be redacted (negative)
NEGATIVE = [
    "The meeting is at 3 PM.",
    "Our version is 1.2.3.4 in production",    # IP-like but also version-ish — regex will match
    "Order ID 12345 for product X",
    "Call me at my desk",
    "See section 3.1.4 of the manual",
]

def was_redacted(text):
    redacted, subs = redact(text)
    return len(subs) > 0, redacted

tp = sum(1 for s in POSITIVE if was_redacted(s)[0])
fn = len(POSITIVE) - tp
fp = sum(1 for s in NEGATIVE if was_redacted(s)[0])
tn = len(NEGATIVE) - fp
print(f"true  positive: {tp}/{len(POSITIVE)}")
print(f"false negative: {fn}/{len(POSITIVE)}   <- PII missed")
print(f"false positive: {fp}/{len(NEGATIVE)}   <- over-redaction")
print(f"true  negative: {tn}/{len(NEGATIVE)}")
for s in NEGATIVE:
    hit, red = was_redacted(s)
    if hit:
        print(f"  -- over-redacted: {s!r}  ->  {red!r}")

true  positive: 6/6
false negative: 0/6   <- PII missed
false positive: 1/5   <- over-redaction
true  negative: 4/5
  -- over-redacted: 'Our version is 1.2.3.4 in production'  ->  'Our version is [IP_a32d6d] in production'


`1.2.3.4` matches the IP pattern — a classic FP. Real pipelines therefore pair the regex with an ML classifier (Presidio, AWS Comprehend) and/or a whitelist of known-safe strings.

## 4. Consent tracking + gated logging

In [5]:
class ConsentStore:
    def __init__(self):
        self.allowed = set()
    def grant(self, user_id):  self.allowed.add(user_id)
    def revoke(self, user_id): self.allowed.discard(user_id)
    def has(self, user_id):    return user_id in self.allowed

@dataclass
class ChatEvent:
    ts: float
    user_id: str
    prompt_redacted: str
    response_redacted: str
    subs_count: int

CHAT_LOG = []   # persistent-ish, in-notebook
consent = ConsentStore()
consent.grant("u1"); consent.grant("u2")
# u3 never granted consent -> logging must skip them

def log_turn(user_id, prompt, response):
    if not consent.has(user_id):
        return "skipped_no_consent"
    rp, subs_p = redact(prompt)
    rr, subs_r = redact(response)
    CHAT_LOG.append(ChatEvent(
        ts=time.time(), user_id=user_id,
        prompt_redacted=rp, response_redacted=rr,
        subs_count=len(subs_p) + len(subs_r),
    ))
    return "logged"

print(log_turn("u1", "email me at a@b.com",  "sure, will do at a@b.com"))
print(log_turn("u2", "call 555-111-2222",     "Will call"))
print(log_turn("u3", "hello",                 "hi"))  # blocked — no consent
print("events logged:", len(CHAT_LOG))

logged
logged
skipped_no_consent
events logged: 2


## 5. Retention policy — drop logs older than N days

In [6]:
# Seed with a few synthetic old events
NOW = time.time()
for user, age_days in [("u1", 180), ("u1", 45), ("u2", 10), ("u2", 400)]:
    CHAT_LOG.append(ChatEvent(
        ts=NOW - age_days * 24 * 3600, user_id=user,
        prompt_redacted="[EMAIL_abc]", response_redacted="ok",
        subs_count=1,
    ))

def apply_retention(logs, max_age_days):
    cutoff = time.time() - max_age_days * 24 * 3600
    kept = [e for e in logs if e.ts >= cutoff]
    dropped = len(logs) - len(kept)
    return kept, dropped

before = len(CHAT_LOG)
CHAT_LOG, dropped = apply_retention(CHAT_LOG, max_age_days=90)
print(f"retention 90 days:  kept {len(CHAT_LOG)} / dropped {dropped} (from {before})")

retention 90 days:  kept 4 / dropped 2 (from 6)


## 6. Right to be forgotten — per-user delete
GDPR Art. 17 requires deleting *every* piece of user data on request. We sweep three stores: chat log, memory/vector store, cache.

In [7]:
# Stand-ins for the three stores
MEMORY = [
    {"user_id": "u1", "content": "User prefers Vietnamese"},
    {"user_id": "u1", "content": "Birthday March 15"},
    {"user_id": "u2", "content": "Espresso drinker"},
]
CACHE = {
    "u1:abc": "cached response 1",
    "u2:def": "cached response 2",
    "u1:xyz": "cached response 3",
}

def forget_user(user_id):
    global CHAT_LOG, MEMORY, CACHE
    removed = {"chat_log": 0, "memory": 0, "cache": 0}
    # Chat log
    n = len(CHAT_LOG)
    CHAT_LOG = [e for e in CHAT_LOG if e.user_id != user_id]
    removed["chat_log"] = n - len(CHAT_LOG)
    # Memory
    n = len(MEMORY)
    MEMORY = [m for m in MEMORY if m["user_id"] != user_id]
    removed["memory"] = n - len(MEMORY)
    # Cache (keys prefixed with user_id)
    before_keys = set(CACHE.keys())
    CACHE = {k: v for k, v in CACHE.items() if not k.startswith(f"{user_id}:")}
    removed["cache"] = len(before_keys) - len(CACHE)
    # Revoke consent on the way out
    consent.revoke(user_id)
    return removed

print("before forget :", {"chat_log": len(CHAT_LOG), "memory": len(MEMORY), "cache": len(CACHE)})
summary = forget_user("u1")
print("removed for u1:", summary)
print("after  forget :", {"chat_log": len(CHAT_LOG), "memory": len(MEMORY), "cache": len(CACHE)})
print("u1 still has consent?", consent.has("u1"))

before forget : {'chat_log': 4, 'memory': 3, 'cache': 3}
removed for u1: {'chat_log': 2, 'memory': 2, 'cache': 2}
after  forget : {'chat_log': 2, 'memory': 1, 'cache': 1}
u1 still has consent? False


## 7. A privacy-aware log schema
Gather the per-user summary of redactions. This answers questions like "how much PII are we redacting per day?" without storing raw PII itself.

In [8]:
# Re-grant consent and re-log a fresh turn to repopulate
consent.grant("u1")
log_turn("u1", "My ID is 123-45-6789 and my card 4111 1111 1111 1111", "got it")
log_turn("u2", "ping me at bob@bob.io", "ok")

rows = [{"user": e.user_id, "ts": e.ts, "subs": e.subs_count} for e in CHAT_LOG]
df = pd.DataFrame(rows)
df["datetime"] = pd.to_datetime(df["ts"], unit="s")
print(df.groupby("user")["subs"].agg(["count", "sum"]).rename(columns={"count": "events", "sum": "redactions"}))

      events  redactions
user                    
u1         1           2
u2         3           3


## 8. Output-side scrub
Even with input redaction, the LLM can accidentally *regurgitate* training-data PII. Scrub the response before showing it to the next user.

In [9]:
ALLOWED_ECHO = {"EMAIL"}   # toy policy: allow the model to echo redacted emails, scrub credit-cards / SSNs

def scrub_outgoing(response):
    redacted, subs = redact(response)
    # Keep the raw value only for kinds we explicitly allow to echo
    parts = []
    restore = {}
    for kind, raw, tok in subs:
        if kind in ALLOWED_ECHO:
            restore[tok] = raw
    for tok, raw in restore.items():
        redacted = redacted.replace(tok, raw)
    return redacted

print(scrub_outgoing("Please reach out to alice@example.com or SSN 123-45-6789."))

Please reach out to alice@example.com or SSN [SSN_a65362].


## 9. Audit trail for every delete
Regulators ask for proof that deletes actually happened. Append-only audit log with timestamp + action + summary.

In [10]:
AUDIT = []

def audited_forget(user_id):
    summary = forget_user(user_id)
    AUDIT.append({
        "ts":       time.time(),
        "actor":    "system",
        "action":   "gdpr_delete",
        "user_id":  user_id,
        "summary":  summary,
    })
    return summary

consent.grant("u2")
log_turn("u2", "ping", "pong")
print(audited_forget("u2"))
print(json.dumps(AUDIT, indent=2, default=str))

{'chat_log': 4, 'memory': 1, 'cache': 1}
[
  {
    "ts": 1776771003.592555,
    "actor": "system",
    "action": "gdpr_delete",
    "user_id": "u2",
    "summary": {
      "chat_log": 4,
      "memory": 1,
      "cache": 1
    }
  }
]


## 10. Quick checks

In [11]:
# Positive cases: each must produce at least one substitution
for s in POSITIVE:
    assert was_redacted(s)[0], s
# Stable hash: the same value always maps to the same token
assert stable_token("EMAIL", "alice@example.com") == stable_token("EMAIL", "ALICE@example.com")
# Retention actually drops old rows
assert dropped >= 1
# forget_user scrubs all three stores
consent.grant("u9")
log_turn("u9", "email at x@x.com", "ok")
MEMORY.append({"user_id": "u9", "content": "note"})
CACHE["u9:k"] = "v"
rem = forget_user("u9")
assert rem["chat_log"] >= 1 and rem["memory"] == 1 and rem["cache"] == 1
# Consent revoked after forget
assert not consent.has("u9")
print("OK — redaction, consent, retention, right-to-forget, audit all behave.")

OK — redaction, consent, retention, right-to-forget, audit all behave.


## Reflection questions

1. Our regex produces a **false positive** on `1.2.3.4`. Sketch a two-layer approach that keeps regex as a fast path and adds an ML checker for the grey zone.
2. `stable_token` uses a salted hash. What happens if you ship the salt in the same logs as the tokens? Why does rotating the salt *break* cross-day correlation?
3. For a right-to-be-forgotten request, we scrubbed three stores. Name two more stores a production system typically also has to sweep.
4. The output scrubber whitelisted `EMAIL` only. Name a scenario where whitelisting **PHONE** is justifiable, and one where it is reckless.

## References
- Source theory: [`9-4-privacy.mdx`](../../llm-quest-theory/level-9/9-4-privacy.mdx)
- Next: [`9-5-ops-boss`](9-5-ops-boss.ipynb) — the Level 9 boss.